# M3L3 E15 — Tool calling con LangGraph
### Módulo 3 · Lecture 3 · Sistemas Multiagente

**Ejercicio paralelo:** E12 (orquestador con OpenAI Functions)

## ¿Qué vas a aprender hoy?
- definir tools con el decorador `@tool` de LangChain.
- usar `llm.bind_tools()` para que el LLM seleccione tools de forma nativa.
- construir el ciclo ReAct: `agent → tool_executor → END`.


## ¿Qué necesitás saber antes?

Venís de E12 donde construiste `ToolSpec` con `name`, `func` y `description`. La lección central era que **las descripciones son prompts**. En E15 el LLM elige la tool usando su capacidad nativa de function-calling.

> **bind_tools():** método que registra una lista de tools en el LLM. Cuando el modelo responde, puede devolver un `tool_call` con el nombre de la tool y los argumentos, en lugar de texto plano.

| E12 Python puro | E15 LangGraph |
|---|---|
| `ToolSpec(name, func, description)` | `@tool` — docstring es la descripción |
| `choose_tool(query, tools)` manual con scoring | `llm.bind_tools([...])` — el LLM decide nativamente |
| Loop manual con `if tool_chosen:` | `add_conditional_edges` cierra el ciclo ReAct |
| Sin visualización del ciclo | `draw_mermaid()` muestra el loop |

El ciclo ReAct en grafo:

```
START
  |
  v
agent  --> (tool_call presente) --> tool_executor --> END
       --> (sin tool_call)      --> END
```


## Paso 1 — Elegí tu proveedor de LLM

> **Nota:** `bind_tools()` requiere que el proveedor soporte function calling. OpenAI, Gemini y Claude 3+ lo soportan.

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

In [ ]:
!pip install langgraph langchain-core -q

from typing import TypedDict
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

print("LangGraph y LangChain Core listos.")

## Sección 1 — El decorador `@tool`

> **@tool:** decorador que convierte una función Python en una herramienta de LangChain. El docstring se convierte en la descripción que el LLM usa con `bind_tools()` para decidir cuándo llamarla.

```python
@tool
def nombre_tool(query: str) -> str:
    """Descripción clara de cuándo usar esta tool.
    El LLM lee esto para decidir si la llama o no."""
    return resultado
```

Las tools devuelven respuestas de la knowledge base. El LLM decide cuál llamar basándose en la descripción.

In [ ]:
knowledge_base = {
    "hr":      ["Vacaciones: 15 días hábiles por año.", "Seguro médico desde el primer día.", "Licencias: pedido en PeopleOps."],
    "tech":    ["VPN: reiniciar cliente y validar MFA.", "Contraseña: restablecer en portal de identidad."],
    "billing": ["Facturas: cargar antes del día 25.", "Reembolsos: adjuntar recibo y centro de costo."],
}

@tool
def hr_tool(query: str) -> str:
    """Responde preguntas sobre vacaciones, licencias, beneficios y seguro médico de empleados."""
    return "HRAgent: " + " | ".join(knowledge_base["hr"])


@tool
def tech_tool(query: str) -> str:
    """Resuelve problemas técnicos: VPN, contraseña, MFA, acceso a sistemas, notebook dañado."""
    return "TechAgent: " + " | ".join(knowledge_base["tech"])


@tool
def billing_tool(query: str) -> str:
    """Gestiona facturas, reembolsos, comprobantes de pago y centros de costo."""
    return "BillingAgent: " + " | ".join(knowledge_base["billing"])


TOOLS = [hr_tool, tech_tool, billing_tool]
TOOLS_MAP = {t.name: t for t in TOOLS}
print("Tools registradas:", list(TOOLS_MAP.keys()))

## Sección 2 — bind_tools y el ciclo ReAct

> **llm.bind_tools(tools):** crea un nuevo LLM que conoce las tools. Cuando se invoca, puede devolver un `AIMessage` con `.tool_calls` — una lista de tools que el modelo decidió llamar.

```python
llm_with_tools = llm.bind_tools([hr_tool, tech_tool, billing_tool])

response = llm_with_tools.invoke([HumanMessage(content="VPN no funciona")])
# response.tool_calls = [{"name": "tech_tool", "args": {"query": "VPN no funciona"}}]
```

El nodo `agent_node` usa `llm_with_tools` para decidir qué tool llamar. Si el modelo devuelve `tool_calls`, escribe el nombre en `tool_call` y los args en `tool_args`. Si no, escribe la respuesta directamente.

**Tu TODO:** implementar `tool_executor` y `should_use_tool`.

In [ ]:
class AgentState(TypedDict):
    query: str
    tool_call: str    # nombre de la tool elegida
    tool_args: dict   # argumentos para la tool
    tool_result: str  # resultado de ejecutar la tool
    response: str     # respuesta final para el usuario


llm_with_tools = llm.bind_tools(TOOLS)


def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke([HumanMessage(content=state["query"])])
    if response.tool_calls:
        tc = response.tool_calls[0]
        return {"tool_call": tc["name"], "tool_args": tc["args"]}
    return {"tool_call": "", "tool_args": {}, "response": response.content}

In [ ]:
def tool_executor(state: AgentState) -> dict:
    # TODO: buscar la tool en TOOLS_MAP con state["tool_call"]
    # Invocarla con TOOLS_MAP[...].invoke(state["tool_args"])
    # Devolver {"tool_result": resultado, "response": resultado}
    return {"tool_result": "", "response": ""}


def should_use_tool(state: AgentState) -> str:
    # TODO: si state["tool_call"] no es vacío → "tool_executor"; si no → END
    return END

## Sección 3 — Construir el ciclo ReAct

El edge condicional después de `agent` implementa la decisión del ReAct:

```
agent  -- tool_call != "" --> tool_executor --> END
       -- tool_call == "" --> END  (respuesta directa del LLM)
```

En un ReAct completo, `tool_executor` volvería a `agent` para que el LLM procese el resultado. Aquí cerramos en `END` para mantener el foco en la estructura.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("agent",         agent_node)
graph.add_node("tool_executor", tool_executor)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_use_tool, {"tool_executor": "tool_executor", END: END})
graph.add_edge("tool_executor", END)

app = graph.compile()
print("Grafo compilado.")

El LLM usa su capacidad nativa de function-calling para elegir la tool correcta — no keywords, no scoring.

In [ ]:
EMPTY = {"query": "", "tool_call": "", "tool_args": {}, "tool_result": "", "response": ""}

for q in ["tengo problemas con la VPN", "¿cuántas vacaciones tengo?", "subir factura", "qué hacemos hoy"]:
    r = app.invoke({**EMPTY, "query": q})
    print(f"[tool={r['tool_call'] or 'ninguna':12s}] {r['response'][:55]}")

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    r1 = app.invoke({**EMPTY, "query": "no puedo conectarme a la VPN"})
    assert r1["tool_call"] == "tech_tool", f"esperaba tech_tool: {r1['tool_call']}"
    assert "TechAgent" in r1["response"],  f"esperaba TechAgent: {r1['response']}"

    r2 = app.invoke({**EMPTY, "query": "quiero pedir vacaciones"})
    assert r2["tool_call"] == "hr_tool",   f"esperaba hr_tool: {r2['tool_call']}"

    print("Checks E15 OK")

run_checks()

## ¿Qué aprendiste hoy?

- `bind_tools()` es la forma oficial de darle herramientas a un LLM: el modelo decide cuál usar con su propio razonamiento, no con un `if/else`.
- `AIMessage.tool_calls` es el mecanismo por el que el LLM comunica su decisión de tool.
- El ciclo ReAct es el patrón central de los agentes con herramientas.

## Próximo ejercicio

En **E16** vas a agregar memoria persistente con `MemorySaver` para que el bot recuerde el contexto entre conversaciones.
